# Head konteks epoch 5: evaluasi test dengan model tetap

Buka dari root **SongMap**, lalu **Run All**. Notebook menjalankan Python sibling **Section-Annotations/.venv-songformer311** secara eksplisit.

Pada validation sebelumnya, head konteks memperoleh akurasi **79,38%**, macro F1 **0,7755**, dan bridge F1 **0,5714** pada epoch 5. Ada 33 kesalahan diperbaiki dan 18 prediksi benar menjadi salah dibanding worship-v3. Kita akan mengukur hasil pada **seluruh 34 lagu test** dengan epoch dan parameter yang sama.

Tiga pembanding: **worship-v3**, **kalibrasi bridge–chorus yang sudah dipilih dari train**, dan **head konteks epoch 5 di atas probabilitas asli worship-v3**. Kalibrator tidak ditumpuk ke head konteks.

Sebelum test, script harus menghasilkan ulang confusion matrix, macro F1, akurasi, dan NLL validation yang tersimpan. Bobot, parameter kalibrasi, runtime, dan seluruh inventory test dikunci dalam `selection.json`, serta checkpoint tambahan disalin ke `frozen-context.pt`. Tidak ada training atau pemilihan epoch baru.


In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, json

SONGMAP_ROOT=Path.cwd().resolve()
assert (SONGMAP_ROOT/'scripts/evaluate_frozen_context.py').is_file(), 'Buka dari root SongMap'
SONGSTRUCT_ROOT=(SONGMAP_ROOT/'../Section-Annotations').resolve()
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'

# Pilih run lengkap dari eksperimen sebelumnya; tidak mencari checkpoint berdasarkan skor test.
files=list((SONGMAP_ROOT/'outputs.local').glob('section-context-*/comparison.json'))
assert files, 'Laporan section_context_accuracy.ipynb belum ditemukan'
CONTEXT_RUN=max(files,key=lambda p:p.stat().st_mtime).parent.resolve()
selected=json.loads((CONTEXT_RUN/'comparison.json').read_text())
run_meta=json.loads((CONTEXT_RUN/'run.json').read_text())
EXPECTED_EPOCH=5
assert selected.get('full_validation') and selected.get('split')=='val'
assert selected['best_epoch']==EXPECTED_EPOCH, f"Run terpilih epoch {selected['best_epoch']}; arahkan CONTEXT_RUN ke run epoch 5 yang sudah dipilih"
assert (CONTEXT_RUN/'best.pt').is_file(), 'Checkpoint head konteks tidak ditemukan'
BASELINE_DEVICE='mps'  # Forward SongFormer tetap memakai cache encoder; tidak menjalankan encoder.
CONTEXT_DEVICE=run_meta['device']  # Gunakan runtime yang sama dengan validation sebelumnya, umumnya cpu.
cache=SONGMAP_ROOT/'outputs.local/section-context-cache-v1'
output=SONGMAP_ROOT/'outputs.local'/('frozen-context-test-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
subprocess.run([str(RUN_PYTHON),'-c',
    "import sys,numpy,torch; print('Python:',sys.executable); print('NumPy:',numpy.__version__,'Torch:',torch.__version__)"],check=True)
print('Model tetap:',CONTEXT_RUN,'| epoch:',EXPECTED_EPOCH)
print('Validation yang harus direproduksi:',selected['candidate']['macro_f1_present'])
print('Output test:',output)


Jalankan cell berikut. Ringkasan validation menggunakan cache kecil yang sudah tersimpan. Forward SongFormer pada test memakai cache fitur encoder asli, lalu hasil probabilitasnya disimpan per lagu. Ringkasan audio untuk test dibuat sekali. MuQ/MusicFM tidak dijalankan ulang.

Jika terputus, jalankan ulang **cell evaluasi berikut dengan variabel output yang sama**. Hasil per lagu yang lengkap akan dipakai kembali. Jika kernel ditutup, arahkan variabel `output` ke folder evaluasi sebelumnya setelah setup. Tidak perlu mengulang training.


In [ ]:
def run_test():
    if (output/'test-all.json').is_file():
        print('Evaluasi sudah selesai:',output/'test-all.json')
        return
    command=[str(RUN_PYTHON),'-u',str(SONGMAP_ROOT/'scripts/evaluate_frozen_context.py'),
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--context-run',str(CONTEXT_RUN),
        '--expected-epoch',str(EXPECTED_EPOCH),'--baseline-device',BASELINE_DEVICE,
        '--device',CONTEXT_DEVICE,'--cache-dir',str(cache),'--output-dir',str(output)]
    log=Path(str(output)+'.log')
    log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=SONGMAP_ROOT,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Gagal ({status}); lihat {log}. Model dan hasil per lagu yang selesai tetap tersedia.')

run_test()


In [ ]:
report=json.loads((output/'test-all.json').read_text())
assert report['full_test'] and report['validation_reproduction_passed']
print('Test:',report['tracks'],'lagu | epoch tetap:',report['selected_epoch'])
for name in ('worship_v3','fixed_calibration','section_context'):
    score=report[name]
    print(f"{name}: accuracy={score['section_accuracy']:.2%}; macro F1={score['macro_f1_present']:.4f}")
print('F1 per kelas (v3 | kalibrasi tetap | konteks):')
for label in report['worship_v3']['label_order']:
    scores=[report[name]['per_class'][label]['f1'] for name in ('worship_v3','fixed_calibration','section_context')]
    recall=report['section_context']['per_class'][label]['recall']
    print(f"  {label}: {scores[0]:.4f} | {scores[1]:.4f} | {scores[2]:.4f}; context recall={recall:.2%}")
for key in ('changed_sections_vs_v3','changed_sections_vs_fixed_calibration'):
    changes=report[key]
    fixed=sum(not s['baseline_correct'] and s['candidate_correct'] for s in changes)
    regressed=sum(s['baseline_correct'] and not s['candidate_correct'] for s in changes)
    print(key,':',len(changes),'berubah;',fixed,'diperbaiki;',regressed,'benar menjadi salah')
print('Kirim file:',output/'test-all.json')
print('Checkpoint snapshot:',report['frozen_context_checkpoint'])
print('Tidak ada training, tuning parameter, atau pemasangan ke aplikasi dari evaluasi ini.')


Bagikan **test-all.json**. Ini perbandingan label pada batas section yang tetap; belum menguji pipeline Core ML terkompresi atau memperbaiki keterlambatan boundary outro.

Split test ini pernah dievaluasi untuk model sebelumnya. Evaluasi sekarang memakai model yang dikunci dari validation dan tidak memakai hasil test untuk memilih ulang epoch/parameter; jangan menafsirkan set yang sama sebagai dataset baru yang belum pernah dilihat.
